# 📊 Dataset Generation — GPU Cooling Digital Twin

**Purpose**: Generate a synthetic GPU telemetry dataset for training the predictive thermal control model.

> ⚠️ **SYNTHETIC DATA** — This dataset does not come from physical hardware. It is generated using realistic relationships between workload, utilization, power, memory, clock, and temperature.

The generated dataset will contain ~500K rows with:
- 8 workload types with distinct telemetry signatures
- Workload transitions and burst events
- Realistic thermal inertia and cooling response
- Future temperature/power labels for supervised learning

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Libraries loaded.')

In [ ]:
# === WORKLOAD PROFILES ===
# Each profile defines base ranges and temporal behavior for GPU metrics

WORKLOAD_PROFILES = {
    'normal': {
        'utilization': (30, 45), 'power': (200, 350), 'memory': (20, 35),
        'clock': (1200, 1400), 'noise': 0.03, 'ramp_speed': 0.1
    },
    'moderate': {
        'utilization': (50, 65), 'power': (400, 550), 'memory': (40, 55),
        'clock': (1400, 1600), 'noise': 0.04, 'ramp_speed': 0.15
    },
    'high_gpu': {
        'utilization': (80, 95), 'power': (700, 900), 'memory': (50, 70),
        'clock': (1600, 1800), 'noise': 0.05, 'ramp_speed': 0.2
    },
    'cnn_training': {
        'utilization': (85, 95), 'power': (750, 850), 'memory': (75, 90),
        'clock': (1500, 1700), 'noise': 0.03, 'ramp_speed': 0.25
    },
    'transformer_inference': {
        'utilization': (60, 80), 'power': (800, 950), 'memory': (85, 95),
        'clock': (1600, 1900), 'noise': 0.06, 'ramp_speed': 0.3
    },
    'cuda_matmul': {
        'utilization': (90, 99), 'power': (850, 950), 'memory': (30, 45),
        'clock': (1800, 2100), 'noise': 0.02, 'ramp_speed': 0.35
    },
    'burst': {
        'utilization': (20, 95), 'power': (300, 950), 'memory': (25, 80),
        'clock': (1200, 2000), 'noise': 0.08, 'ramp_speed': 0.5
    },
    'idle': {
        'utilization': (5, 15), 'power': (80, 150), 'memory': (5, 15),
        'clock': (800, 1100), 'noise': 0.02, 'ramp_speed': 0.05
    }
}

print(f'Defined {len(WORKLOAD_PROFILES)} workload profiles.')

In [ ]:
# === THERMAL SIMULATION PARAMETERS ===

C_THERMAL = 50.0        # J/°C - thermal capacitance
T_AMBIENT = 25.0        # °C
T_COOLANT_SUPPLY = 30.0 # °C
CP_COOLANT = 1.8        # kJ/(kg·°C) - dielectric coolant
COOLANT_DENSITY = 1.6   # kg/L
MAX_FLOW = 10.0         # L/min
DT = 1.0                # seconds per timestep

def compute_cooling_power(flow_rate, valve_pos, t_gpu, t_coolant):
    """Compute cooling power from flow and temperatures."""
    # Convert flow from L/min to kg/s
    mass_flow = (flow_rate / 60.0) * COOLANT_DENSITY
    # Valve effectiveness 0-1
    eta = valve_pos / 100.0
    # Cooling power
    q_cool = mass_flow * CP_COOLANT * 1000 * (t_gpu - t_coolant) * eta
    return max(0, q_cool)

def thermal_step(t_gpu, p_gpu, flow_rate, valve_pos, t_coolant):
    """Single thermal simulation step."""
    q_cool = compute_cooling_power(flow_rate, valve_pos, t_gpu, t_coolant)
    # Thermal equation
    dt_gpu = (DT / C_THERMAL) * (p_gpu - q_cool)
    t_gpu_new = t_gpu + dt_gpu
    # Clamp to physical range
    t_gpu_new = np.clip(t_gpu_new, T_AMBIENT, 105.0)
    return t_gpu_new

print('Thermal model defined.')

In [ ]:
# === GENERATE DATASET ===

NUM_SAMPLES = 500_000
PREDICTION_HORIZON = 30  # steps (seconds)

records = []
workload_names = list(WORKLOAD_PROFILES.keys())

# Start conditions
t_gpu = 45.0
t_coolant = T_COOLANT_SUPPLY
flow_rate = 3.0
valve_pos = 40.0
current_workload = 'normal'
workload_time = 0
workload_duration = 300  # how long each workload lasts

# Smooth transition targets
target_util = 40.0
target_power = 300.0
target_mem = 25.0
target_clock = 1300.0

util_smooth = 40.0
power_smooth = 300.0
mem_smooth = 25.0
clock_smooth = 1300.0

print(f'Generating {NUM_SAMPLES:,} samples...')

for i in range(NUM_SAMPLES + PREDICTION_HORIZON):
    # Switch workload periodically
    workload_time += 1
    if workload_time >= workload_duration:
        workload_time = 0
        current_workload = np.random.choice(workload_names)
        workload_duration = np.random.randint(100, 600)
        profile = WORKLOAD_PROFILES[current_workload]
        target_util = np.random.uniform(*profile['utilization'])
        target_power = np.random.uniform(*profile['power'])
        target_mem = np.random.uniform(*profile['memory'])
        target_clock = np.random.uniform(*profile['clock'])
    
    profile = WORKLOAD_PROFILES[current_workload]
    alpha = profile['ramp_speed'] * DT * 0.1
    noise = profile['noise']
    
    # Smooth transitions with noise
    util_smooth += alpha * (target_util - util_smooth) + np.random.normal(0, noise * target_util)
    power_smooth += alpha * (target_power - power_smooth) + np.random.normal(0, noise * target_power)
    mem_smooth += alpha * (target_mem - mem_smooth) + np.random.normal(0, noise * target_mem)
    clock_smooth += alpha * (target_clock - clock_smooth) + np.random.normal(0, noise * target_clock)
    
    # Add burst patterns
    if current_workload == 'burst':
        burst_phase = (workload_time / max(workload_duration, 1)) * 2 * np.pi
        burst_factor = 0.5 * (1 + np.sin(burst_phase * 3))
        power_smooth = 300 + burst_factor * 650
        util_smooth = 20 + burst_factor * 75
    
    # Sine wave variations for realism
    t = i * DT
    sine_var = np.sin(t * 0.01) * 10 + np.sin(t * 0.05) * 5
    
    gpu_util = np.clip(util_smooth + sine_var * 0.1, 0, 100)
    gpu_power = np.clip(power_smooth + sine_var * 2, 80, 950)
    gpu_mem = np.clip(mem_smooth + sine_var * 0.05, 0, 100)
    gpu_clock = np.clip(clock_smooth + sine_var * 5, 600, 2200)
    
    # Simple reactive controller for data generation
    if t_gpu > 80:
        valve_pos = min(100, valve_pos + 2)
        flow_rate = min(MAX_FLOW, flow_rate + 0.3)
    elif t_gpu > 70:
        valve_pos = min(80, valve_pos + 0.5)
        flow_rate = min(7.0, flow_rate + 0.1)
    elif t_gpu < 60:
        valve_pos = max(20, valve_pos - 0.5)
        flow_rate = max(2.0, flow_rate - 0.1)
    
    # Update coolant temperature
    t_coolant = T_COOLANT_SUPPLY + 0.1 * (t_gpu - T_COOLANT_SUPPLY)
    
    # Thermal step
    t_gpu = thermal_step(t_gpu, gpu_power, flow_rate, valve_pos, t_coolant)
    
    # Cooling demand
    cooling_demand = np.clip(valve_pos / 100.0 * flow_rate / MAX_FLOW, 0, 1)
    
    records.append({
        'timestamp': i,
        'workload': current_workload,
        'gpu_utilization': round(gpu_util, 2),
        'gpu_power': round(gpu_power, 2),
        'gpu_memory': round(gpu_mem, 2),
        'gpu_clock': round(gpu_clock, 2),
        'gpu_temperature': round(t_gpu, 2),
        'coolant_temperature': round(t_coolant, 2),
        'coolant_flow_rate': round(flow_rate, 2),
        'valve_position': round(valve_pos, 2),
        'cooling_demand': round(cooling_demand, 4)
    })
    
    if (i + 1) % 100_000 == 0:
        print(f'  Generated {i+1:,} / {NUM_SAMPLES + PREDICTION_HORIZON:,} samples')

df = pd.DataFrame(records)
print(f'\nDataFrame shape: {df.shape}')
print(df.head())

In [ ]:
# === ADD FUTURE LABELS ===

df['future_temperature_30s'] = df['gpu_temperature'].shift(-PREDICTION_HORIZON)
df['future_power_30s'] = df['gpu_power'].shift(-PREDICTION_HORIZON)

# Drop rows without future labels
df = df.dropna().reset_index(drop=True)
print(f'Dataset with labels: {df.shape}')
print(f'Temperature range: {df["gpu_temperature"].min():.1f}°C to {df["gpu_temperature"].max():.1f}°C')
print(f'Power range: {df["gpu_power"].min():.1f}W to {df["gpu_power"].max():.1f}W')
print(f'\nWorkload distribution:')
print(df['workload'].value_counts())

In [ ]:
# === FEATURE ENGINEERING ===

# Rolling statistics
df['power_rolling_mean_10'] = df['gpu_power'].rolling(10, min_periods=1).mean()
df['power_rolling_max_10'] = df['gpu_power'].rolling(10, min_periods=1).max()
df['util_rolling_mean_10'] = df['gpu_utilization'].rolling(10, min_periods=1).mean()

# Slopes (rate of change)
df['temperature_slope_5'] = df['gpu_temperature'].diff(5) / 5.0
df['power_slope_5'] = df['gpu_power'].diff(5) / 5.0
df['memory_trend_5'] = df['gpu_memory'].diff(5) / 5.0
df['clock_trend_5'] = df['gpu_clock'].diff(5) / 5.0

# Workload encoding
workload_dummies = pd.get_dummies(df['workload'], prefix='wl')
df = pd.concat([df, workload_dummies], axis=1)

# Drop NaN from rolling
df = df.dropna().reset_index(drop=True)
print(f'Final dataset: {df.shape}')
print(f'Features: {list(df.columns)}')

In [ ]:
# === SAVE DATASET ===

# Save to Google Drive (if mounted) or local
try:
    from google.colab import drive
    drive.mount('/content/drive')
    save_path = '/content/drive/MyDrive/gpu_cooling_dataset.parquet'
except:
    save_path = 'gpu_cooling_dataset.parquet'

df.to_parquet(save_path, index=False)
print(f'Dataset saved to: {save_path}')
print(f'File size: {pd.io.common.file_exists(save_path)}')
print(f'\n=== SYNTHETIC DATA GENERATION COMPLETE ===')
print(f'Samples: {len(df):,}')
print(f'Features: {len(df.columns)}')
print(f'Workload types: {df["workload"].nunique()}')